# Generative AI 018 — Tools

Custom tools live in `langchain_core.tools` and run offline. Several common
claims about them turn out to be wrong — each is checked here.

| Part | What we check |
|---|---|
| A | what the model is sent: **219 characters**, no function body |
| B | `@tool` **validates** — it differs from StructuredTool only in constraints |
| C | the docstring is required; `Field(required=True)`; the wrong import |
| D | BaseTool, async on every tool, and a toolkit |

Needs `langchain-core`, `pydantic`.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

## Part A — A tool, and what the model sees

In [ ]:
import json
from langchain_core.tools import tool
from langchain_core.runnables import Runnable
from langchain_core.utils.function_calling import convert_to_openai_tool

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b

print(multiply.name, "|", multiply.description, "|", multiply.args)
print(multiply.invoke({"a": 3, "b": 5}))          # 15
print(isinstance(multiply, Runnable))             # True

# What the MODEL is sent when this tool is attached:
print(json.dumps(convert_to_openai_tool(multiply)))
# {"type": "function", "function": {"name": "multiply",
#   "description": "Multiply two numbers",
#   "parameters": {"properties": {"a": {"type": "integer"}, "b": {"type": "integer"}},
#                  "required": ["a", "b"], "type": "object"}}}
#
# Not one character of the function body. A name, a description and a
# schema are the model's entire view of the tool - which is why the
# docstring and the type hints are the interface, not decoration.

In [ ]:
sent = json.dumps(convert_to_openai_tool(multiply))
assert "return a * b" not in sent            # the body is never sent
print(len(sent), "characters - name, description and schema only")

## Part B — Is @tool 'loose'?

In [ ]:
from typing import Annotated
from pydantic import BaseModel, Field
from langchain_core.tools import StructuredTool

class MultiplyInput(BaseModel):
    a: int = Field(gt=0, description="The first number, positive")
    b: int = Field(description="The second number")

structured = StructuredTool.from_function(
    func=lambda a, b: a * b, name="multiply",
    description="Multiply two numbers", args_schema=MultiplyInput)

def attempt(t, payload):
    try:
        return t.invoke(payload)
    except Exception as e:
        return type(e).__name__

for payload in ({"a": "3", "b": 5}, {"a": "three", "b": 5},
                {"a": 3.7, "b": 2}, {"a": 3}, {"a": -3, "b": 5}):
    print(f"{str(payload):<24} @tool: {str(attempt(multiply, payload)):<17} "
          f"structured: {attempt(structured, payload)}")

# {'a': '3', 'b': 5}       @tool: 15                structured: 15
# {'a': 'three', 'b': 5}   @tool: ValidationError   structured: ValidationError
# {'a': 3.7, 'b': 2}       @tool: ValidationError   structured: ValidationError
# {'a': 3}                 @tool: ValidationError   structured: ValidationError
# {'a': -3, 'b': 5}        @tool: -15               structured: ValidationError
#
# @tool VALIDATES - it builds a Pydantic model from the type hints. Only the
# last row differs: a hint can say "int", not "a positive int". The real
# difference is CONSTRAINTS. And @tool can take those too:

@tool
def safe_multiply(a: Annotated[int, Field(gt=0)], b: int) -> int:
    """Multiply two numbers; the first must be positive."""
    return a * b

print(attempt(safe_multiply, {"a": -3, "b": 5}))   # ValidationError

In [ ]:
assert attempt(multiply, {"a": "three", "b": 5}) == "ValidationError"
assert attempt(multiply, {"a": 3.7, "b": 2}) == "ValidationError"
assert attempt(multiply, {"a": -3, "b": 5}) == -15            # no constraint
assert attempt(structured, {"a": -3, "b": 5}) == "ValidationError"
print("only the constraint differs - and Annotated gives @tool that too")

## Part C — Three things the usual examples get wrong

In [ ]:
import warnings

# ---- 1. the docstring is REQUIRED, not merely recommended ------------------
try:
    @tool
    def no_doc(a: int) -> int:
        return a
except ValueError as e:
    print("ValueError:", e)
# Function must have a docstring if description not provided.

# ---- 2. Field(required=True) is a Pydantic v1 habit ------------------------
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    class OldStyle(BaseModel):
        a: int = Field(required=True, description="The first number")
print(caught[0].category.__name__)                     # PydanticDeprecatedSince20
print(OldStyle.model_json_schema()["properties"]["a"])
# {'description': ..., 'required': True, 'title': 'A', 'type': 'integer'}
#                        ^^^^^^^^^^^^^^ junk, sent to the model
# In v2 a field is required by having no default. Drop required=True.

# ---- 3. StructuredTool is not in langchain.tools ---------------------------
import langchain.tools as lt
print([n for n in ("tool", "StructuredTool", "BaseTool") if hasattr(lt, n)])
# ['tool', 'BaseTool']   - import all three from langchain_core.tools

## Part D — BaseTool, async, and a toolkit

In [ ]:
import asyncio
from langchain_core.tools import BaseTool

class MultiplyTool(BaseTool):
    name: str = "multiply"
    description: str = "Multiply two numbers"
    args_schema: type[BaseModel] = MultiplyInput

    def _run(self, a: int, b: int) -> int:     # must be named _run
        return a * b

t = MultiplyTool()
print(t.invoke({"a": 3, "b": 5}))                    # 15
print(asyncio.run(t.ainvoke({"a": 3, "b": 5})))      # 15 - no _arun needed

@tool
async def async_multiply(a: int, b: int) -> int:
    """Multiply two numbers, asynchronously."""
    return a * b
print(asyncio.run(async_multiply.ainvoke({"a": 3, "b": 5})))   # 15

# Every tool has ainvoke, and @tool takes async functions directly. What
# BaseTool adds is a NATIVE _arun beside _run, and full control of the rest.

# ---- a toolkit is a list of tools behind one method -------------------------
@tool
def add(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b

class MathToolkit:
    def get_tools(self):
        return [add, multiply]

for t in MathToolkit().get_tools():
    print(t.name, t.invoke({"a": 6, "b": 7}))    # add 13, multiply 42

## What to take away

- The model sees a **name, a description and a schema** — never the body.
- `@tool` **validates**; the real difference from StructuredTool is constraints.
- The docstring is **required**. `Field(required=True)` is deprecated.
  `StructuredTool` imports from `langchain_core.tools`.
- **Every tool** has `ainvoke`.

## Exercises

1. Give `multiply` a vague docstring ("does maths"). Print what the model is
   sent. How would a model decide between it and an `add` tool?
2. Write a `divide` tool that rejects `b == 0` using a constraint, not an `if`.
3. Add `return_direct=True` to a tool and look up what it changes for an agent.
4. Write a toolkit of three string tools (upper, reverse, word count) and print
   the schema a model would receive for each.